# Development-only exploratory TabICLv2 evaluation

This notebook renders aggregate results from the completed standalone TabICLv2 experiment. It does not load participant data, fit models, access the previously evaluated test cohort, or alter deployment artifacts.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'reports').exists():
    ROOT = ROOT.parent
TABICL_REPORT = ROOT / 'reports' / 'generated' / 'development-tabiclv2-model-comparison-report.json'
HISTORICAL_REPORT = ROOT / 'reports' / 'generated' / 'development-additional-model-comparison-report.json'
tabicl = json.loads(TABICL_REPORT.read_text(encoding='utf-8'))
historical = json.loads(HISTORICAL_REPORT.read_text(encoding='utf-8'))
assert tabicl['status'] == 'development_only_exploratory_tabiclv2_completed'
assert tabicl['development_participants'] == 500
assert tabicl['final_test_evaluated'] is False
assert tabicl['cross_validation']['shared_fold_plan_sha256'] == historical['cross_validation']['shared_fold_plan_sha256']
print('Verified aggregate reports loaded; no training or participant-level data access performed.')

Verified aggregate reports loaded; no training or participant-level data access performed.


## Method and frozen configuration

Three separate classifiers used `tabicl==2.2.0` and the verified `tabicl-classifier-v2-20260212.ckpt` checkpoint. The experiment preserved the categorical checkpoint plus 15 ordinal baseline answers, five-fold cross-validation repeated five times, three inner folds for calibration and threshold selection, and Average Precision as the primary metric. Inner contexts contained approximately 266 participants, below TabICLv2's reported pretraining range.

In [2]:
domain_labels = {
    'target_communication_concern': 'Communication',
    'target_cognitive_concern': 'Cognitive',
    'target_social_emotional_concern': 'Social-emotional',
}
metric_columns = [
    ('average_precision_pr_auc', 'PR-AUC'), ('roc_auc', 'AUROC'),
    ('accuracy', 'Accuracy'), ('balanced_accuracy', 'Balanced accuracy'),
    ('sensitivity', 'Sensitivity'), ('specificity', 'Specificity'),
    ('precision', 'PPV'), ('negative_predictive_value', 'NPV'),
    ('f1', 'F1'), ('brier_score', 'Brier'),
]
rows = []
for key, label in domain_labels.items():
    domain = tabicl['domains'][key]
    summary = domain['repeat_metric_summary']
    row = {'Domain': label, 'Model': 'TabICLv2'}
    for metric, heading in metric_columns:
        row[heading] = summary[metric]['mean']
    row['PR-AUC 95% descriptive interval'] = str(summary['average_precision_pr_auc']['descriptive_confidence_interval_95'])
    rows.append(row)
performance = pd.DataFrame(rows)
display(performance.style.format({heading: '{:.4f}' for _, heading in metric_columns}))

,Domain,Model,PR-AUC,AUROC,Accuracy,Balanced accuracy,Sensitivity,Specificity,PPV,NPV,F1,Brier,PR-AUC 95% descriptive interval
0,Communication,TabICLv2,0.5207,0.8526,0.7552,0.7420,0.7246,0.7595,0.3004,0.9522,0.4224,0.0785,"[0.5004962293501073, 0.5409492707864069]"
1,Cognitive,TabICLv2,0.5780,0.9055,0.8000,0.8233,0.8528,0.7937,0.3303,0.9786,0.4756,0.0616,"[0.5445339280977459, 0.6114425726722479]"
2,Social-emotional,TabICLv2,0.6338,0.8921,0.8552,0.8460,0.8339,0.8580,0.4405,0.9748,0.5763,0.0643,"[0.6047443211981248, 0.6627588820841283]"


In [3]:
comparison_rows = []
for key, label in domain_labels.items():
    domain = tabicl['domains'][key]
    reference = domain['historical_selected_reference']
    comparison = domain['numerical_comparison_to_historical_selected_model']
    comparison_rows.append({
        'Domain': label,
        'Historical selected model': reference,
        'TabICLv2 PR-AUC': comparison['average_precision_pr_auc']['tabiclv2_mean'],
        'Historical PR-AUC': comparison['average_precision_pr_auc']['historical_reference_mean'],
        'PR-AUC difference': comparison['average_precision_pr_auc']['tabiclv2_minus_reference'],
        'TabICLv2 balanced accuracy': comparison['balanced_accuracy']['tabiclv2_mean'],
        'Historical balanced accuracy': comparison['balanced_accuracy']['historical_reference_mean'],
        'TabICLv2 Brier': comparison['brier_score']['tabiclv2_mean'],
        'Historical Brier': comparison['brier_score']['historical_reference_mean'],
        'Paired interval available': domain['paired_uncertainty_available'],
    })
comparison_table = pd.DataFrame(comparison_rows)
display(comparison_table.style.format({
    'TabICLv2 PR-AUC': '{:.4f}', 'Historical PR-AUC': '{:.4f}', 'PR-AUC difference': '{:+.4f}',
    'TabICLv2 balanced accuracy': '{:.4f}', 'Historical balanced accuracy': '{:.4f}',
    'TabICLv2 Brier': '{:.4f}', 'Historical Brier': '{:.4f}',
}))

,Domain,Historical selected model,TabICLv2 PR-AUC,Historical PR-AUC,PR-AUC difference,TabICLv2 balanced accuracy,Historical balanced accuracy,TabICLv2 Brier,Historical Brier,Paired interval available
0,Communication,xgboost_regularized_balanced,0.5207,0.4958,+0.0249,0.7420,0.7243,0.0785,0.0810,False
1,Cognitive,logistic_l2_balanced,0.5780,0.5386,+0.0394,0.8233,0.8123,0.0616,0.0672,False
2,Social-emotional,xgboost_regularized_balanced,0.6338,0.5866,+0.0472,0.8460,0.7912,0.0643,0.0698,False


## Eight-model development rankings

In [4]:
ranking_rows = []
for key, label in domain_labels.items():
    for rank, item in enumerate(tabicl['domains'][key]['eight_model_pr_auc_ranking'], start=1):
        ranking_rows.append({'Domain': label, 'Rank': rank, 'Candidate': item['candidate'], 'Mean PR-AUC': item['mean_average_precision'], 'Source': item['source']})
rankings = pd.DataFrame(ranking_rows)
display(rankings.style.format({'Mean PR-AUC': '{:.4f}'}))

,Domain,Rank,Candidate,Mean PR-AUC,Source
0,Communication,1,tabiclv2_default_eight_view,0.5207,current_tabiclv2_evaluation
1,Communication,2,xgboost_regularized_balanced,0.4958,historical_seven_model_report
2,Communication,3,catboost_ordered_balanced,0.4740,historical_seven_model_report
3,Communication,4,logistic_l2_balanced,0.4721,historical_seven_model_report
4,Communication,5,ebm_main_effects_balanced,0.4307,historical_seven_model_report
5,Communication,6,random_forest_balanced_leaf5,0.4226,historical_seven_model_report
6,Communication,7,extra_trees_balanced_leaf5,0.4081,historical_seven_model_report
7,Communication,8,svm_rbf_balanced,0.3271,historical_seven_model_report
8,Cognitive,1,tabiclv2_default_eight_view,0.5780,current_tabiclv2_evaluation
9,Cognitive,2,logistic_l2_balanced,0.5386,historical_seven_model_report


## Thresholds, confusion matrices and calibration

In [5]:
threshold_rows, confusion_rows, calibration_rows = [], [], []
for key, label in domain_labels.items():
    domain = tabicl['domains'][key]
    stability = domain['threshold_stability']
    threshold_rows.append({'Domain': label, **stability})
    totals = {name: 0 for name in ('true_negative', 'false_positive', 'false_negative', 'true_positive')}
    for repeat in domain['repeat_results']:
        for name, value in repeat['metrics']['confusion_matrix'].items():
            totals[name] += value
    confusion_rows.append({'Domain': label, **totals, 'Interpretation': 'pooled over five repeated validations'})
    for bin_row in domain['pooled_repeated_validation_calibration_bins']:
        calibration_rows.append({'Domain': label, **bin_row})
display(Markdown('### Experimental threshold distribution'))
display(pd.DataFrame(threshold_rows).style.format({name: '{:.4f}' for name in ('mean', 'standard_deviation', 'minimum', 'maximum')}))
display(Markdown('### Confusion counts'))
display(pd.DataFrame(confusion_rows))
display(Markdown('### Pooled repeated-validation calibration bins'))
display(pd.DataFrame(calibration_rows).style.format({'mean_predicted_probability': '{:.4f}', 'observed_positive_rate': '{:.4f}'}))

### Experimental threshold distribution

,Domain,mean,standard_deviation,minimum,maximum
0,Communication,0.1371,0.0518,0.0601,0.2368
1,Cognitive,0.1136,0.0468,0.0567,0.2317
2,Social-emotional,0.1582,0.0361,0.0762,0.2529


### Confusion counts

,Domain,true_negative,false_positive,false_negative,true_positive,Interpretation
0,Communication,1667,528,84,221,pooled over five repeated validations
1,Cognitive,1774,461,39,226,pooled over five repeated validations
2,Social-emotional,1892,313,49,246,pooled over five repeated validations


### Pooled repeated-validation calibration bins

,Domain,lower_bound,upper_bound,count,mean_predicted_probability,observed_positive_rate
0,Communication,0.000000,0.100000,1522,0.0305,0.0361
1,Communication,0.100000,0.200000,565,0.1407,0.0991
2,Communication,0.200000,0.300000,143,0.2489,0.2937
3,Communication,0.300000,0.400000,99,0.3408,0.3636
4,Communication,0.400000,0.500000,74,0.4489,0.5541
5,Communication,0.500000,0.600000,32,0.5492,0.8750
6,Communication,0.600000,0.700000,18,0.6468,0.7222
7,Communication,0.700000,0.800000,23,0.7463,0.7826
8,Communication,0.800000,0.900000,17,0.8513,0.7647
9,Communication,0.900000,1.000000,7,0.9389,0.4286


In [6]:
config = tabicl['tabiclv2']
runtime = pd.DataFrame([{
    'Package': tabicl['runtime_versions']['tabicl'],
    'Checkpoint': config['checkpoint'],
    'Checkpoint SHA-256': config['checkpoint_sha256'],
    'Views': config['n_estimators'],
    'Device': config['device'],
    'Fine-tuning': config['fine_tuning'],
    'Runtime minutes': tabicl['runtime_seconds'] / 60,
    'Outer evaluations': 3 * tabicl['cross_validation']['outer_evaluations_per_domain'],
}])
display(runtime.style.format({'Runtime minutes': '{:.2f}'}))

,Package,Checkpoint,Checkpoint SHA-256,Views,Device,Fine-tuning,Runtime minutes,Outer evaluations
0,2.2.0,45c13f180e7197fbc11b24a9cf5e3be795902967bcc61e2223dc19245559a934,BDC7DBD5E4FF21F8F0456FCF90C6B7CDF72DBEA960F2D05B19BEC19F9B3D4ED0,8,cpu,False,34.52,75


## Explainability and limitations

TabICLv2 offers optional SHAP compatibility using feature masking, but all-fold SHAP was intentionally not run. Base-model SHAP values would explain the uncalibrated model output and would not directly decompose the subsequent sigmoid-calibrated probability. Explanations are associations, not clinical or causal evidence.

The same 500-person development cohort has been reused across research stages. The five repeated-CV estimates share participants, and the historical aggregate report does not retain aligned participant predictions. Consequently, descriptive intervals and numerical differences cannot establish paired statistical superiority. No final fitted TabICLv2 model exists, and the previously evaluated 125-person cohort was not accessed.

In [7]:
lines = ['## Scientific conclusion', '']
for row in comparison_rows:
    lines.append(f"- **{row['Domain']}**: TabICLv2 was numerically higher in development PR-AUC by {row['PR-AUC difference']:+.4f} versus {row['Historical selected model']}.")
lines.extend([
    '',
    'TabICLv2 ranked first numerically in the eight-model development comparison for all three domains. These are exploratory point-estimate results, not evidence of reliable superiority, because aligned paired predictions are unavailable and the development cohort has been reused adaptively. The existing selected models should therefore remain unchanged pending separate review and genuinely independent validation.'
])
display(Markdown('\n'.join(lines)))

## Scientific conclusion

- **Communication**: TabICLv2 was numerically higher in development PR-AUC by +0.0249 versus xgboost_regularized_balanced.
- **Cognitive**: TabICLv2 was numerically higher in development PR-AUC by +0.0394 versus logistic_l2_balanced.
- **Social-emotional**: TabICLv2 was numerically higher in development PR-AUC by +0.0472 versus xgboost_regularized_balanced.

TabICLv2 ranked first numerically in the eight-model development comparison for all three domains. These are exploratory point-estimate results, not evidence of reliable superiority, because aligned paired predictions are unavailable and the development cohort has been reused adaptively. The existing selected models should therefore remain unchanged pending separate review and genuinely independent validation.